# MarketingOS — Transcrição de reuniões com GPU

Transcreve todos os áudios de um cliente com `faster-whisper`, preserva português, gera arquivos individuais e uma transcrição consolidada no padrão `clients/[slug]/inputs/meetings/`.

Antes de executar: no Colab, selecione **Ambiente de execução → Alterar tipo de ambiente → GPU**.

In [ ]:
import subprocess, sys
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
if gpu.returncode != 0:
    raise RuntimeError('GPU não detectada. Ative uma GPU no ambiente de execução do Colab.')
print('GPU disponível:', gpu.stdout.strip())


In [ ]:
!pip -q install faster-whisper
!apt-get -qq update && apt-get -qq install -y ffmpeg
print('Dependências instaladas.')


## Configuração

Use `drive` quando o MarketingOS estiver sincronizado no Google Drive. Use `upload` para selecionar áudios diretamente do computador.

In [ ]:
CLIENT_SLUG = 'fortunato'
MEETING_DATE = '2026-07-17'
SOURCE_MODE = 'upload'  # pronto para usar agora; troque por 'drive' quando sincronizar o projeto
DRIVE_PROJECT_ROOT = '/content/drive/MyDrive/marketing-os'
MODEL = 'large-v3'  # produção; troque por 'small' para um teste rápido
LANGUAGE = 'pt'
BEAM_SIZE = 5
print({'cliente': CLIENT_SLUG, 'data': MEETING_DATE, 'modelo': MODEL, 'idioma': LANGUAGE})


In [ ]:
from pathlib import Path
import shutil

if SOURCE_MODE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    project_root = Path(DRIVE_PROJECT_ROOT)
    input_dir = project_root / 'clients' / CLIENT_SLUG / 'inputs' / 'audio'
    output_dir = project_root / 'clients' / CLIENT_SLUG / 'inputs' / 'meetings' / f'whisper-{MEETING_DATE}-gpu'
else:
    from google.colab import files
    input_dir = Path('/content/marketingos-audio')
    output_dir = Path('/content/marketingos-transcricoes')
    input_dir.mkdir(parents=True, exist_ok=True)
    uploaded = files.upload()
    for name, data in uploaded.items():
        (input_dir / name).write_bytes(data)

output_dir.mkdir(parents=True, exist_ok=True)
extensions = {'.ogg', '.mp3', '.wav', '.m4a', '.mp4', '.webm', '.aac', '.flac'}
audio_files = sorted(p for p in input_dir.iterdir() if p.suffix.lower() in extensions)
if not audio_files:
    raise FileNotFoundError(f'Nenhum áudio encontrado em {input_dir}')
print(f'{len(audio_files)} arquivos encontrados em {input_dir}')
for p in audio_files:
    print('-', p.name)


In [ ]:
from faster_whisper import WhisperModel

model = WhisperModel(MODEL, device='cuda', compute_type='float16')
print(f'Modelo {MODEL} carregado na GPU.')


In [ ]:
import json
from datetime import datetime

results = []
for index, audio_path in enumerate(audio_files, start=1):
    print(f'[{index}/{len(audio_files)}] {audio_path.name}')
    segments_iter, info = model.transcribe(
        str(audio_path),
        language=LANGUAGE,
        beam_size=BEAM_SIZE,
        vad_filter=True,
        word_timestamps=True,
        condition_on_previous_text=True,
    )
    segments = []
    words = []
    transcript_parts = []
    for segment in segments_iter:
        text = segment.text.strip()
        if not text:
            continue
        transcript_parts.append(text)
        segments.append({'start': round(segment.start, 3), 'end': round(segment.end, 3), 'text': text})
        for word in segment.words or []:
            words.append({
                'id': f'w{len(words)}',
                'text': word.word.strip(),
                'start': round(word.start, 3),
                'end': round(word.end, 3),
                'probability': round(word.probability, 4),
            })
    transcript = ' '.join(transcript_parts)
    payload = {
        'source': audio_path.name,
        'model': MODEL,
        'language': info.language,
        'language_probability': round(info.language_probability, 4),
        'duration_seconds': round(info.duration, 3),
        'segments': segments,
        'words': words,
    }
    (output_dir / f'{audio_path.stem}.txt').write_text(transcript, encoding='utf-8')
    (output_dir / f'{audio_path.stem}.json').write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    results.append({'audio': audio_path.name, 'text': transcript, 'duration': info.duration})

print(f'Transcrição concluída: {len(results)} arquivos.')


In [ ]:
header = [
    f'# Transcrição consolidada — {CLIENT_SLUG} — {MEETING_DATE}',
    '',
    f'> Gerada com faster-whisper `{MODEL}` em GPU, idioma `{LANGUAGE}`.',
    '> Revise nomes próprios, marcas, valores e termos técnicos antes da análise estratégica.',
    '',
]
body = []
for item in results:
    body.extend([f"## {item['audio']}", '', item['text'], ''])
consolidated = '\n'.join(header + body)
consolidated_path = output_dir.parent / f'{MEETING_DATE}-transcript-gpu.md'
consolidated_path.write_text(consolidated, encoding='utf-8')
manifest = {
    'client_slug': CLIENT_SLUG,
    'meeting_date': MEETING_DATE,
    'model': MODEL,
    'language': LANGUAGE,
    'source_dir': str(input_dir),
    'output_dir': str(output_dir),
    'files': len(results),
    'total_duration_seconds': round(sum(r['duration'] for r in results), 2),
    'generated_at': datetime.now().isoformat(),
}
(output_dir / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print('Consolidado:', consolidated_path)
print(json.dumps(manifest, ensure_ascii=False, indent=2))


In [ ]:
# Download opcional quando SOURCE_MODE == 'upload'
if SOURCE_MODE == 'upload':
    from google.colab import files
    archive = shutil.make_archive('/content/transcricoes-marketingos', 'zip', output_dir)
    files.download(archive)
else:
    print('Arquivos salvos diretamente no Google Drive.')


## Próximo passo no MarketingOS

Use o arquivo `YYYY-MM-DD-transcript-gpu.md` como entrada da skill `/inteligencia reuniao [slug]`. Ela deve gerar `YYYY-MM-DD-signals.json`, separando fatos, inferências, dores, desejos, objeções e hipóteses de aquisição.